In [1]:
import matlab.engine
eng = matlab.engine.start_matlab("-desktop")

In [2]:
eng.addpath(r'C:\Users\ua24606\git_2\Sprint0626')  # Add MATLAB function folder

'C:\\Users\\ua24606\\AppData\\Roaming\\MathWorks\\MATLAB Add-Ons\\Toolboxes\\pim\\pim-packages\\dcrg-bristol\\baff\\Examples;C:\\Users\\ua24606\\AppData\\Roaming\\MathWorks\\MATLAB Add-Ons\\Toolboxes\\pim\\pim-packages\\dcrg-bristol\\baff\\tbx;C:\\Users\\ua24606\\AppData\\Roaming\\MathWorks\\MATLAB Add-Ons\\Toolboxes\\pim\\pim-packages\\dcrg-bristol\\Laca\\examples;C:\\Users\\ua24606\\AppData\\Roaming\\MathWorks\\MATLAB Add-Ons\\Toolboxes\\pim\\pim-packages\\dcrg-bristol\\Laca\\tbx;C:\\Users\\ua24606\\MATLAB;C:\\Users\\ua24606\\git_2\\aeroelastic_examples\\tbx;C:\\Users\\ua24606\\git\\awi_wing_nonlinear\\nbbd\\tbx;C:\\Users\\ua24606\\git\\awi_wing_nonlinear\\nbbd\\dpd;C:\\Users\\ua24606\\git\\awi_wing_nonlinear\\nbbd\\tbx\\nbbd;C:\\Users\\ua24606\\git\\awi_wing_nonlinear\\nbbd\\dpd\\MatlabAutoDiff;C:\\Users\\ua24606\\git\\awi_wing_nonlinear\\nbbd\\dpd\\MatlabAutoDiff\\.github;C:\\Users\\ua24606\\git\\awi_wing_nonlinear\\nbbd\\dpd\\MatlabAutoDiff\\.github\\workflows;C:\\Users\\ua24606\\

In [5]:
data = [4, 0.63, 23, 0.85]
matlab_data = eng.physical_model_indep_sweep_ar_he(matlab.double(data))
# matlab_data = eng.BayesOptObjective_bf_nastran_for_multi_fidelity(matlab.double(data))

In [3]:
"multi-fidelity optimisation"
import warnings

warnings.filterwarnings("ignore")
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plt

In [4]:
def HF_function(x):
    return eng.BayesOptObjective_bf_nastran_for_multi_fidelity(matlab.double(x))
def LF_function(x):
    return eng.BayesOptObjective_bf_for_multi_fidelity(matlab.double(x))

In [5]:
from smt.applications.mfk import NestedLHS

In [6]:
# Dimension
ndim = 3

#  Nested LHS sampling

xlimits = np.array([[0, 45], [0.45, 0.9], [11, 23]])
xdoes = NestedLHS(nlevel=2, xlimits=xlimits)
Xt_c, Xt_e = xdoes(4)

# Evaluate the HF and LF functions
yt_e = HF_function(Xt_e)
yt_c = LF_function(Xt_c)

In [7]:
from smt.applications import MFK

In [10]:
yt_ccc = np.array(yt_c)
yt_eee = np.array(yt_e)

In [11]:
yt_c = yt_ccc.reshape(-1, 1)
yt_e = yt_eee.reshape(-1, 1)

In [12]:
# %% Build the MFK object
sm_MF = MFK()

# low-fidelity dataset names being integers from 0 to level-1
sm_MF.set_training_values(Xt_c, yt_c, name=0)
# high-fidelity dataset without name
sm_MF.set_training_values(Xt_e, yt_e)

# train the model
sm_MF.train()

___________________________________________________________________________
   
                                    MFK
___________________________________________________________________________
   
 Problem size
   
      # training points.        : 4
   
___________________________________________________________________________
   
 Training
   
   Training ...
   Training - done. Time (sec):  0.3124433


In [19]:
import scipy as sp
from scipy.stats import norm
from scipy.optimize import minimize
from smt.surrogate_models import KRG

In [13]:
def ME(lvl, sigma_delta, sigma_rho, theta_vol_lvl, costs, last_val=0):
    """
    ME : Model enhancement criterion
    """

    MSE_r = last_val + sigma_delta[lvl] * np.prod(sigma_rho[lvl:]) * theta_vol_lvl[lvl]
    if last_val == 0:
        return True, MSE_r

    return np.log(MSE_r) - np.log(last_val) > 2.0 * np.log(
        1 + costs[lvl] / costs[lvl - 1]
    ), MSE_r

In [14]:
class MFEGO:
    """
    Attributes :
        functions, functions from lowest to highest level
    """

    def __init__(
        self, functions, kriging_model, bounds, costs, sol, multi_start=20, eps=1e-3
    ):
        self.km = kriging_model
        self.bounds = bounds
        self.n_lvls = 2
        if self.n_lvls != len(functions):
            raise Exception(
                "There must be as many functions as fidelity levels in your Kriging model"
            )
        if self.n_lvls != len(costs):
            raise Exception(
                "There must be as many costs as fidelity levels in your Kriging model"
            )

        self.levels = [0, None]  ## Highest level has the name None for smt
        self.functions = {0: functions[0], None: functions[1]}
        self.doe_x = {
            0: self.km.training_points[0][0][0],
            None: self.km.training_points[None][0][0],
        }
        self.doe_y = {
            0: self.km.training_points[0][0][1],
            None: self.km.training_points[None][0][1],
        }
        self.dim = self.km.nx
        self.costs = np.array(costs) + 0.0
        self.gain = []
        self.cost_hist = []
        self.corr_fun = self.km.options["corr"]
        self.multi_start = multi_start
        self.y0_best = np.min(self.doe_y[None])
        self.sol = sol
        self.eps = eps

    def cost_increment(self, lvls_added):
        # computing the cost increment after each iteration
        # lvls_added list of booleans, True if to be added, False otherwise
        return np.sum(lvls_added * self.costs) / self.costs[-1]

    def compute_EI(self, x):
        # compute expected improvement at point x
        x = np.atleast_2d(x)
        Y_min = np.min(self.km.training_points[None][0][1])
        y_pred = self.km.predict_values(x)
        mse = self.km.predict_variances(x)
        y_pred = np.atleast_2d(y_pred)
        sigma_y = np.sqrt(mse)
        y_normed = (Y_min - y_pred[:, 0]) / sigma_y
        EI = (Y_min - y_pred[:, 0]) * sp.stats.norm.cdf(
            y_normed
        ) + sigma_y * sp.stats.norm.pdf(y_normed)

        return -np.ravel(EI)

    def add_level(self, x):
        MSE, sigma_rho = self.km.predict_variances_all_levels(x)

        sigma_delta = np.zeros((self.n_lvls,))
        sigma_rho = np.array(sigma_rho)
        theta_vol_lvl = np.ones((self.n_lvls,))
        flags = self.n_lvls * [False]
        for lvl in self.levels:
            if lvl is None:
                lvl = self.n_lvls - 1

            theta_vol_lvl[lvl] = 1.0
            if lvl == 0:
                sigma_delta[lvl] = MSE[0, 0]
                flags[lvl], val = ME(
                    lvl, sigma_delta, sigma_rho, theta_vol_lvl, self.costs
                )
            else:
                sigma_delta[lvl] = MSE[0, lvl] - (MSE[0, lvl - 1]) * sigma_rho[lvl - 1]
                flags[lvl], val = ME(
                    lvl, sigma_delta, sigma_rho, theta_vol_lvl, self.costs, last_val=val
                )
            if not flags[lvl]:
                break

        return flags

    def run(self, budget):
        self.current_cost = 0
        n_iter = 0
        epsilon = 1
        doe_init_size_HF = len(self.doe_y[None])
        doe_init_size_LF = len(self.doe_y[0])
        init_cost = len(sm.training_points[0][0][1]) / self.costs[-1] + len(
            sm.training_points[None][0][1]
        )
        while self.current_cost < budget and epsilon >= self.eps:
            n_iter = n_iter + 1
            # optimization of the EI
            candidat = []
            value = []
            X0 = np.random.rand(self.multi_start, self.dim)

            for i in range(len(self.bounds)):
                X0[:, i] = (
                    X0[:, i] * (-self.bounds[i, 0] + self.bounds[i, 1])
                    + self.bounds[i, 0]
                )
            # Multi start
            for x in X0:
                new_point = sp.optimize.fmin_slsqp(
                    self.compute_EI, np.atleast_2d(x), bounds=self.bounds, iprint=0
                )
                candidat.append(new_point)
                value.append(self.compute_EI(new_point))
            value = np.array(value)

            ind_min = value.argmin()
            x_n = candidat[ind_min].reshape((1, self.dim))
            # Evaluation of the true function(s)
            levels_to_be_added = self.add_level(x_n)

            for lvl in self.levels:
                ll = lvl if lvl is not None else -1
                if levels_to_be_added[ll]:  ## level infill criterion
                    y_n = self.functions[lvl](x_n)
                    y_n = np.atleast_2d(y_n)
                    # updating the kriging meta-model
                    self.doe_x[lvl] = np.insert(self.doe_x[lvl], 0, x_n, axis=0)
                    self.doe_y[lvl] = np.insert(self.doe_y[lvl], 0, y_n, axis=0)
                    self.km.set_training_values(
                        self.doe_x[lvl], self.doe_y[lvl], name=lvl
                    )
            # retraining the model
            self.km.train()

            # computing cost increment
            self.current_cost = self.current_cost + self.cost_increment(
                levels_to_be_added
            )

            # saving data for visualization and postprocesing
            self.cost_hist.append(self.current_cost)
            self.gain.append(self.y0_best - np.min(self.doe_y[None]))
            epsilon = np.abs(np.min(self.doe_y[None]) - self.sol)
            print("######################################")
            print("iteration n=", n_iter)
            print(
                "Number of Hi Fi calls during optimization= ",
                len(self.doe_y[None]) - doe_init_size_HF,
            )
            print(
                "Number of Low Fi calls during optimization= ",
                len(self.doe_y[0]) - doe_init_size_LF,
            )
            print("max EI found at x=", x_n)
            print(
                "Current best value=",
                np.min(self.doe_y[None]),
                " distance to reference solution: ",
                epsilon,
            )
            print("value of the EI =", value.min())
            print(
                "Enrichement current spent budget: ",
                self.current_cost,
                " Total cost including DOE: ",
                self.current_cost + init_cost,
            )
            # add total budget + dif 1D/1D

        Y_min = np.min(self.doe_y[None])
        return Y_min, self.km, self.doe_x, self.doe_y

In [22]:
import numpy as np
import matplotlib.pyplot as plt
from smt.applications import MFK

from smt.applications.mfk import NestedLHS

dim = 3
n = 4  # number of high fidelity points (number of low fi is twice)
nlevel = 2
# ub0 = 1.0
# lb0 = 0.0

xlimits = np.array([[0, 45], [0.45, 0.9], [11, 23]])

costs = np.array([1, 100])  # low and high-fidelity costs
# initial_cost = np.sum(np.array(n)*costs)

# np.random.seed(1)
# To create the nested DOE
# xdoes = NestedLHS(nlevel=nlevel, xlimits=xlimits)
# Xc, Xe = xdoes(n)


# yc = LF_function(Xc)
# ye = HF_function(Xe)

# time.sleep(5)

sm = MFK(theta0=Xt_e.shape[1] * [1.0], print_global=False)
sm.set_training_values(Xt_c, yt_c, name=0)
sm.set_training_values(Xt_e, yt_e)
sm.train()

bounds = np.array([[0, 45], [0.45, 0.9], [11, 23]])
algo_MFEGO = MFEGO([LF_function, HF_function], sm, bounds, costs=costs, sol=1.7728e4)
# Running the EGO algorithm for n_iter
budget = 30
Y_min, sm, doe_x, doe_y = algo_MFEGO.run(budget)

target_level = algo_MFEGO.levels[-1]

######################################
iteration n= 1
Number of Hi Fi calls during optimization=  0
Number of Low Fi calls during optimization=  1
max EI found at x= [[ 0.          0.59140253 23.        ]]
Current best value= 17934.036518334866  distance to reference solution:  206.03651833486583
value of the EI = -621.2721612068847
Enrichement current spent budget:  0.01  Total cost including DOE:  4.09
######################################
iteration n= 2
Number of Hi Fi calls during optimization=  0
Number of Low Fi calls during optimization=  2
max EI found at x= [[ 0.          0.59736767 20.58237634]]
Current best value= 17934.036518334866  distance to reference solution:  206.03651833486583
value of the EI = -504.47106239145455
Enrichement current spent budget:  0.02  Total cost including DOE:  4.1
######################################
iteration n= 3
Number of Hi Fi calls during optimization=  0
Number of Low Fi calls during optimization=  3
max EI found at x= [[9.58141741e-06 5

Optimization failed. Try increasing the 'nugget' above its current value of 2.220446049250313e-14.


ValueError: array must not contain infs or NaNs